In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
store = pd.read_csv("store.csv")

C:\Users\cw\AppData\Local\Temp\ipykernel_8788\1336641893.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("train.csv")


In [5]:
train = train.merge(store, on="Store", how="left")
test = test.merge(store, on="Store", how="left")

In [6]:
train['Date'] = pd.to_datetime(train['Date'])
test['Date'] = pd.to_datetime(test['Date'])

In [7]:
for df in [train, test]:
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week
    df['DayOfWeek'] = df['Date'].dt.dayofweek

In [8]:
train = train[train['Open'] == 1]

In [9]:
categorical_cols = ['StoreType', 'Assortment', 'StateHoliday']
train = pd.get_dummies(train, columns=categorical_cols)
test = pd.get_dummies(test, columns=categorical_cols)

In [10]:
train, test = train.align(test, join='left', axis=1)

In [11]:
X = train.drop(columns=['Sales', 'Customers', 'Date'])
y = train['Sales']

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

In [13]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

In [14]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5
}

In [ ]:
# from lightgbm import early_stopping, log_evaluation

# model = lgb.train(
#     params,
#     train_data,
#     valid_sets=[val_data],
#     num_boost_round=2000,
#     callbacks=[early_stopping(100), log_evaluation(100)]
# )

In [20]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

model = LGBMRegressor(
    objective='regression',
    learning_rate=0.05,
    n_estimators=2000,
    num_leaves=31,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[early_stopping(100), log_evaluation(100)]
)


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: PromoInterval: object